<a href="https://colab.research.google.com/github/axelr004/AD_2026Q4_Grupo20/blob/main/numpy2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Librerias

In [1]:
import pandas as pd
import numpy as np

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Pandas: 2.2.3
NumPy: 2.4.6


Cargar los datos de Alexaander

In [2]:
URL_MED = "https://raw.githubusercontent.com/axelr004/AD_2026Q4_Grupo20/main/AE3/datos/medellin.csv"

URL_MAD = "https://raw.githubusercontent.com/axelr004/AD_2026Q4_Grupo20/main/AE3/datos/madrid.csv"

medellin = pd.read_csv(URL_MED)
madrid = pd.read_csv(URL_MAD)

display(medellin.head())
display(madrid.head())

,fecha,temperatura_max_c,precipitacion_mm
0,2023-01-01,23.8,14.5
1,2023-01-02,26.2,7.9
2,2023-01-03,23.5,9.6
3,2023-01-04,25.1,9.0
4,2023-01-05,25.7,6.9


,fecha,temperatura_max_c,precipitacion_mm
0,2023-01-01,14.8,0.0
1,2023-01-02,12.3,1.0
2,2023-01-03,11.8,0.0
3,2023-01-04,12.3,0.0
4,2023-01-05,12.5,0.0


Revisar los datos

In [3]:
for nombre, df in [("Medellín", medellin), ("Madrid", madrid)]:
    print("\n", nombre)
    print("Dimensiones:", df.shape)
    print("Valores faltantes:")
    print(df.isna().sum())
    print("Fechas duplicadas:", df["fecha"].duplicated().sum())


 Medellín
Dimensiones: (365, 3)
Valores faltantes:
fecha                0
temperatura_max_c    0
precipitacion_mm     0
dtype: int64
Fechas duplicadas: 0

 Madrid
Dimensiones: (365, 3)
Valores faltantes:
fecha                0
temperatura_max_c    0
precipitacion_mm     0
dtype: int64
Fechas duplicadas: 0


Preparar las fechas y variables

In [4]:
for df in [medellin, madrid]:
    df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")

    df["temperatura_max_c"] = pd.to_numeric(
        df["temperatura_max_c"], errors="coerce"
    )

    df["precipitacion_mm"] = pd.to_numeric(
        df["precipitacion_mm"], errors="coerce"
    )

Unir las dos ciudades

In [5]:
comparacion = pd.merge(
    medellin,
    madrid,
    on="fecha",
    how="inner",
    suffixes=("_med", "_mad"),
    validate="one_to_one"
)

columnas = [
    "temperatura_max_c_med",
    "precipitacion_mm_med",
    "temperatura_max_c_mad",
    "precipitacion_mm_mad"
]

comparacion = comparacion.dropna(subset=columnas).copy()

print("Fechas comparables:", len(comparacion))
display(comparacion.head())

Fechas comparables: 365


,fecha,temperatura_max_c_med,precipitacion_mm_med,temperatura_max_c_mad,precipitacion_mm_mad
0,2023-01-01,23.8,14.5,14.8,0.0
1,2023-01-02,26.2,7.9,12.3,1.0
2,2023-01-03,23.5,9.6,11.8,0.0
3,2023-01-04,25.1,9.0,12.3,0.0
4,2023-01-05,25.7,6.9,12.5,0.0


Crear arreglos de NumPy

In [6]:
fechas = comparacion["fecha"].to_numpy()

temp_med = comparacion["temperatura_max_c_med"].to_numpy(dtype=float)
temp_mad = comparacion["temperatura_max_c_mad"].to_numpy(dtype=float)

prec_med = comparacion["precipitacion_mm_med"].to_numpy(dtype=float)
prec_mad = comparacion["precipitacion_mm_mad"].to_numpy(dtype=float)

print("Cantidad de temperaturas Medellín:", len(temp_med))
print("Cantidad de temperaturas Madrid:", len(temp_mad))

Cantidad de temperaturas Medellín: 365
Cantidad de temperaturas Madrid: 365


Indexacion y slicing

In [7]:
print("Primera temperatura de Medellín:", temp_med[0])
print("Primeros 10 días de Medellín:", temp_med[:10])
print("Temperaturas de Madrid entre los días 10 y 20:", temp_mad[10:20])

Primera temperatura de Medellín: 23.8
Primeros 10 días de Medellín: [23.8 26.2 23.5 25.1 25.7 25.2 23.7 23.5 23.8 22.2]
Temperaturas de Madrid entre los días 10 y 20: [10.6 10.5 11.  10.2  9.6 10.4 10.   6.6  8.3 13.5]


Mascaras booleanas

In [8]:
lluvia_med = prec_med > 0.1
lluvia_mad = prec_mad > 0.1

calor_med = temp_med > 25
calor_mad = temp_mad > 25

print("Días lluviosos Medellín:", np.count_nonzero(lluvia_med))
print("Días lluviosos Madrid:", np.count_nonzero(lluvia_mad))

print("Días sobre 25 °C Medellín:", np.count_nonzero(calor_med))
print("Días sobre 25 °C Madrid:", np.count_nonzero(calor_mad))

Días lluviosos Medellín: 342
Días lluviosos Madrid: 99
Días sobre 25 °C Medellín: 319
Días sobre 25 °C Madrid: 136


Broadcasting

In [9]:
matriz_temperaturas = np.column_stack((temp_med, temp_mad))

medias = np.mean(matriz_temperaturas, axis=0, keepdims=True)

temperaturas_centradas = matriz_temperaturas - medias

print("Medias por ciudad:", medias)
print("Primeros cinco resultados:")
print(temperaturas_centradas[:5])

Medias por ciudad: [[27.53315068 21.88767123]]
Primeros cinco resultados:
[[ -3.73315068  -7.08767123]
 [ -1.33315068  -9.58767123]
 [ -4.03315068 -10.08767123]
 [ -2.43315068  -9.58767123]
 [ -1.83315068  -9.38767123]]


Estadisticas descriptivas

In [10]:
for nombre, temperaturas, precipitaciones in [
    ("Medellín", temp_med, prec_med),
    ("Madrid", temp_mad, prec_mad)
]:
    print("\n", nombre)
    print("Temperatura media:", np.mean(temperaturas))
    print("Temperatura mínima:", np.min(temperaturas))
    print("Temperatura máxima:", np.max(temperaturas))
    print("Precipitación acumulada:", np.sum(precipitaciones))
    print("Días lluviosos:", np.count_nonzero(precipitaciones > 0.1))


 Medellín
Temperatura media: 27.533150684931503
Temperatura mínima: 21.3
Temperatura máxima: 32.8
Precipitación acumulada: 2782.0
Días lluviosos: 342

 Madrid
Temperatura media: 21.887671232876713
Temperatura mínima: 5.9
Temperatura máxima: 39.3
Precipitación acumulada: 437.19999999999993
Días lluviosos: 99
